# SAM3 — bands and bands+edge only

Two runs, five frozen training volumes, 25 updates each, same 96 validation volumes.

| Run | Bands target | Edge target |
|---|---:|---:|
| Bands | 10% | — |
| Bands + edge | 10% | 10% |

There is no baseline task. The comparison cell reads the completed 25-update baseline
from `results_5vol_24conditions_v1` (65.10% validation Dice), without training or evaluating it again.
Targets use the existing sequential gradient calibration; the p95 ceiling is five times each target.
The fold, seed, learning-rate horizon (200), warmups (20), sampling and no-augmentation settings are unchanged.

After the old run has stopped cleanly, upload `sam3_geofwi_two_runs_code.zip` to the existing
`MyDrive/hippo_sam3_geofwi_exploration/input` folder and run this notebook in order.
Reuse the existing data ZIP and base weights. Mount Drive as **filippofocaccia2@gmail.com**.
Results go to `results_5vol_25updates_bands10_edge10_v1`; old results are preserved.
At most two workers run, subject to GPU memory and throughput checks.


In [1]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED=time.monotonic()
HOME_DIR=Path('/content/drive/MyDrive/hippo_sam3_geofwi_exploration')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_5vol_25updates_bands10_edge10_v1'
BUNDLE=Path('/content/hippo_sam3_geofwi_two_runs_bundle')
BASE_WEIGHTS=Path('/content/drive/MyDrive/hippo_medsam3_msd_extension/input/sam3.pt')
# Change BASE_WEIGHTS if your existing sam3.pt is elsewhere; it is never overwritten.
RESULTS.mkdir(parents=True,exist_ok=True)
print('Results and recovery:',RESULTS)


Mounted at /content/drive
Results and recovery: /content/drive/MyDrive/hippo_sam3_geofwi_exploration/results_5vol_25updates_bands10_edge10_v1


## Stage verified inputs on the local SSD
Files already present with matching hashes are reused. Different files cause a stop; active work is preserved.
The data ZIP contains only the exact five training and 96 validation volumes, not the full dataset.


In [2]:
EXPECTED_UPLOADS={'sam3_geofwi_101vol_data.zip': '094456253ed3bf63e792d08b53eeb0c8bd36fc577a857c6cc6557d6f8271c5f0', 'sam3.pt': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e', 'sam3_geofwi_two_runs_code.zip': '3c7ea0c2f8b95d1dff0bd14d937f7d0238f8379b81ca07dc7fb1643a2b1f4536'}
EXPECTED_MANIFEST='7a8904c0151702fced8c1a922a760efe9f2c7b80afe0f599c1f2cd34b2612684'

import fcntl, shutil, zipfile
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def safe(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe ZIP path')
    return p
def install_zip(path,expected,allowed):
    if sha(path)!=expected:raise ValueError('ZIP hash mismatch: '+str(path))
    with zipfile.ZipFile(path) as archive:
        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError('Duplicate ZIP members')
        if set(archive.namelist())!=set(allowed):raise ValueError('ZIP inventory mismatch')
        for info in archive.infolist():
            safe(info.filename)
            if (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Symlink in ZIP')
        for name,expected_sha in allowed.items():
            target=BUNDLE/safe(name)
            if target.exists():
                if target.is_symlink() or sha(target)!=expected_sha:raise ValueError('Existing file differs: '+str(target))
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.staging')
            with archive.open(name) as source,tmp.open('wb') as dest:shutil.copyfileobj(source,dest)
            if sha(tmp)!=expected_sha:raise ValueError('Extracted member differs')
            tmp.replace(target)
(BUNDLE/'work').mkdir(parents=True,exist_ok=True)
with (BUNDLE/'work/sweep.lock').open('a') as lock:
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    package=INPUT/'sam3_geofwi_two_runs_code.zip'
    if sha(package)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Code ZIP changed')
    with zipfile.ZipFile(package) as archive:
        raw=archive.read('bundle_manifest.json')
        if hashlib.sha256(raw).hexdigest()!=EXPECTED_MANIFEST:raise ValueError('Scientific manifest differs')
        files=json.loads(raw);files['bundle_manifest.json']=EXPECTED_MANIFEST
    install_zip(package,EXPECTED_UPLOADS[package.name],files)
    data_files={'data/'+n:h for n,h in json.loads((BUNDLE/'data_manifest.json').read_text()).items()}
    if any(not (BUNDLE/n).exists() or sha(BUNDLE/n)!=h for n,h in data_files.items()):
        install_zip(INPUT/'sam3_geofwi_101vol_data.zip',EXPECTED_UPLOADS['sam3_geofwi_101vol_data.zip'],data_files)
    base=BUNDLE/'weights/sam3.pt';base.parent.mkdir(exist_ok=True)
    if not base.exists():
        if shutil.disk_usage(BUNDLE).free<16*1024**3:raise RuntimeError('Need16 GiB free runtime SSD before staging')
        if sha(BASE_WEIGHTS)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Existing sam3.pt differs')
        tmp=base.with_suffix('.staging');shutil.copyfile(BASE_WEIGHTS,tmp)
        if sha(tmp)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Checkpoint copy differs')
        tmp.replace(base)
    elif sha(base)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Local SAM3 checkpoint differs')
    provenance=RESULTS/'source'
    provenance.mkdir(exist_ok=True)
    dest=provenance/package.name
    if not dest.exists():shutil.copyfile(package,dest)
    if sha(dest)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Durable source package differs')
print('Verified frozen source,101 volumes and SAM3 weights on local SSD.')


Verified frozen source,101 volumes and SAM3 weights on local SSD.


## Environment and commands
Reuses matching installed versions; otherwise installs the same pinned dependencies as the MSD notebook.
An A100/H100-class BF16 GPU with at least 40 GB VRAM is required. Colab speed and CUDA equivalence
are checked on your actual GPU; local packaging tests cannot establish runtime speedups.


In [3]:
VENV=Path('/content/hippo_sam3_geofwi_venv');PYTHON=VENV/'bin/python'
def setup(args):
    p=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (RESULTS/'setup.log').open('a') as log:log.write(p.stdout)
    print('\n'.join(p.stdout.splitlines()[-30:]))
    if p.returncode:raise RuntimeError('Setup failed; see setup.log')
if not PYTHON.exists():setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    path=Path('/content/sam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',path)
    setup([PYTHON,path])
probe="import importlib.metadata as m,sys; from pathlib import Path; lines=Path(sys.argv[1]).read_text().splitlines(); ok=all(m.version(s.split('==')[0])==s.split('==')[1] if '==' in s else bool(m.version(s)) for s in lines if s.strip()); sys.exit(0 if ok else 1)"
if subprocess.run([str(PYTHON),'-c',probe,str(BUNDLE/'requirements-colab.txt')],capture_output=True).returncode:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:print('Reusing matching dependency versions')
def execute(args,log_name,parallel=False):
    p=subprocess.Popen(list(map(str,args)),start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
                code=p.wait();break
            except KeyboardInterrupt:
                print('Clean stop requested. Keep this VM connected until the cell returns and recovery is verified.')
                if p.poll() is None:
                    if parallel:p.send_signal(signal.SIGUSR1)
                    else:os.killpg(p.pid,signal.SIGINT)
    if code:raise RuntimeError('Command failed; see '+str(RESULTS/log_name))
def campaign(command):
    execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS],command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)



    Found existing installation: torch 2.11.0+cu130
    Not uninstalling torch at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_geofwi_venv
    Can't uninstall 'torch'. No files were found to uninstall.
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Not uninstalling tokenizers at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_geofwi_venv
    Can't uninstall 'tokenizers'. No files were found to uninstall.
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Not uninstalling transformers at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_geofwi_venv
    Can't uninstall 'transformers'. No files were found to uninstall.
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu130
    Not uninstalling torchvision at /usr/local/lib/python3.13/dist-packages, outside environmen

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Shared GPU calibration — once per matching runtime
Runs the original native/bands/edge backward checks, then measures the three-loss gradient Gram matrix
on exactly20 slabs from the five training volumes. Each variant derives its own sequential median/p95
coefficients using the unchanged calibrator; these raw measurements do not depend on stopping updates
or requested target. No historical cluster coefficients are imported. Partial calibration resumes.
A passed preflight is reused only for identical source, data, base weights, GPU and software.


In [4]:
campaign('preflight')


httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.2.attn.qkv
  - backbone.vision_backbone.trunk.blocks.2.attn.proj
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc2
  - backbone.vision_backbone.t

## Run or resume
`MAX_WORKERS=2` adapts to available VRAM, host RAM, CPU and observed throughput. Startup is staggered;
a lower limit lets existing workers finish. Each worker owns its model, optimizer, RNG and run folder.
Only bands and bands+edge are scheduled; there is no baseline task. Every30 seconds the cell reports
GPU utilization, workers, completed runs, updates, evaluation volume and slice.

Deterministic CPU transforms are cached with a1 GiB bound per worker and released for evaluation.
This binary problem has one prompt: no two-class image-sharing speedup is assumed. Evaluation-only
text reuse requires identical labels, strict log-probability agreement, unchanged RNG and at least2%
observed slice-time improvement on the first full volume; the first slice of every later volume is checked.
Failed checks use the original evaluator. Every slice and all101 volumes remain in the evaluation.

Local checkpoints are atomic after every update; coherent Drive snapshots upload asynchronously about
every3 minutes. **Interrupt once and wait** for clean-stop recovery to finish. Completed volumes and a
partial current volume are retained on clean stop. A hard VM loss with charged updates newer than saved
recovery stops for an audit rather than silently replaying updates. Finished metrics, curves and logs are
verified on Drive before their checkpoints are removed. Keep this Colab tab/runtime connected.

Only one VM may write this RESULTS folder. The local lock prevents overlapping cells on one VM; it is
not a cross-VM lock. A manual RESULTS/STOP file is respected. The22-hour budget includes setup in this
notebook; reduce it if the VM was already in use. Colab may interrupt earlier.


In [5]:
MAX_WORKERS=2
MAX_NEW_RUNS=2
SESSION_HOURS=22.0
FAST_EVALUATION=True
remaining=SESSION_HOURS-(time.monotonic()-NOTEBOOK_STARTED)/3600
if remaining<=0.1:raise RuntimeError('Session budget exhausted; resume in a fresh matching runtime')
args=[PYTHON,BUNDLE/'parallel.py','--bundle',BUNDLE,'--durable',RESULTS,
      '--max-workers',str(MAX_WORKERS),'--max-runs',str(MAX_NEW_RUNS),'--hours',str(remaining)]
if FAST_EVALUATION:args.append('--fast-evaluation')
execute(args,'parallel_run.log',parallel=True)


Preserved 0 verified completed runs. 2 verified task identities.
Evaluation feature reuse is enabled only after a checkpoint-specific numerical and speed check. Training remains unchanged.
GPU 0% | VRAM 0.0/80.0 GiB | workers 0/1 | archived 0/2
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
RUN GeoFWI3D_SAM3_5vol_dice_bands_edge_u25_b10_e100_f0s0 — resume update 0/25, evaluated 0 volumes
GPU 0% | VRAM 3.9/80.0 GiB | workers 1/1 | archived 0/2
  GeoFWI3D_SAM3_5vol_dice_bands_edge_u25_b10_e100_f0s0: loading, update 0, evaluated 0 volumes
GPU 0% | VRAM 7.6/80.0 GiB | workers 1/1 | archived 0/2
  GeoFWI3D_SAM3_5vol_dice_bands_edge_u25_b10_e100_f0s0: training, update 0, evaluated 0 volumes
GPU 0% | VRAM 10.3/80.0 GiB | workers 1/1 | archived 0/2
  GeoFWI3D_SAM3_5vol_dice_bands_edge_u25_b10_e100_f0s0: tr

## Compare completed results
Gains use the saved 25-update Colab baseline and the new bands10 run. The baseline is read from its existing archive; it is never rerun.
The JSON also includes paired volume wins/losses, p10 and worst Dice change, FN/FP, and historical results.
Incomplete controls leave gains blank. This is a single-fold/seed exploration.


In [6]:
execute([PYTHON,BUNDLE/'runtime.py','report','--bundle',BUNDLE,'--durable',RESULTS,'--baseline-results',HOME_DIR/'results_5vol_24conditions_v1'],'report.log')
import pandas as pd
result=json.loads((RESULTS/'comparison.json').read_text())
table=pd.DataFrame(result['runs'])
if not table.empty:
    table['validation_dice_percent']=100*table['validation_dice']
    columns=['updates','scientific_arm','bands_target','edge_target','validation_dice_percent',
             'gain_vs_native_pp','gain_vs_bands10_pp','foreground_fn','foreground_fp']
    display(table[columns].sort_values(['updates','scientific_arm','bands_target','edge_target']))
    table.to_csv(RESULTS/'comparison.csv',index=False)
print(result['new_completed'],'/',result['new_total'],'complete')


Archived 2/2 runs


,updates,scientific_arm,bands_target,edge_target,validation_dice_percent,gain_vs_native_pp,gain_vs_bands10_pp,foreground_fn,foreground_fp
1,25,dice_bands,0.1,NaN,68.414490,3.312317,0.000000,3242533,561362
0,25,dice_bands_edge,0.1,0.1,51.998523,-13.103651,-16.415968,4913341,246404


2 / 2 complete
